# Cardiovascular Disease Prediction
## Notebook 03: Support Vector Machine (SVM)
---
**Author:** MS26912370 (Member 2)
**Algorithm:** Support Vector Machine with RBF Kernel

### Algorithm Background

A **Support Vector Machine (SVM)** is a supervised learning algorithm that finds the
optimal decision boundary (**hyperplane**) that maximises the **margin** between classes.

#### Key Concepts:
- **Maximum Margin Classifier:** The hyperplane is positioned to maximise the distance
  to the nearest training points (the **support vectors**).
- **Kernel Trick:** For non-linearly separable data, SVM maps features into a
  higher-dimensional space using a **kernel function**.
  - **RBF (Radial Basis Function):** `K(x,z) = exp(-γ||x-z||²)` — maps to infinite dimensions.
- **Regularisation (C):** Controls the trade-off between maximising margin and minimising
  training errors. Low C = wider margin (more regularisation); High C = narrower margin.
- **Gamma (γ):** Defines the influence radius of a single training example.
  Low γ = large radius (smooth boundary); High γ = small radius (complex boundary).

#### Why SVM for this Dataset?
1. Effective in **high-dimensional spaces** even with many features
2. **Memory efficient** — uses only support vectors for the decision function
3. **Kernel trick** captures non-linear patterns in cardiovascular risk factors
4. Provides **probability calibration** via Platt scaling (`probability=True`)
5. Contrasts well with Random Forest for academic comparison

#### SVM vs Random Forest — Key Differences
| Aspect              | Random Forest          | SVM                        |
|--------------------|------------------------|----------------------------|
| Approach           | Ensemble of trees      | Single optimal hyperplane  |
| Feature scaling    | Not required           | **Required** (done in NB01)|
| Interpretability   | Feature importance     | Support vectors            |
| Training speed     | Fast                   | Slower for large N         |
| Kernel trick       | No                     | Yes (RBF)                  |

In [23]:
# Import all required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
import warnings

from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve, ConfusionMatrixDisplay
)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
os.makedirs('plots', exist_ok=True)

print("✓ Libraries imported successfully.")

✓ Libraries imported successfully.


In [24]:
import os

# ============================================================
# ENVIRONMENT SETUP — comment out the block you are NOT using
# ============================================================

# ── Google Colab Method ──────────────────────────────────
# Mounts Drive and changes to your project folder so
# preprocessed_data.pkl (saved by NB01) can be found here.
# from google.colab import drive
# drive.mount('/content/drive')
# DRIVE_PATH = '/content/drive/MyDrive/ML-Assignment/attemp2'
# os.chdir(DRIVE_PATH)
# print(f"✓ Working directory: {DRIVE_PATH}")

# ── Local Jupyter Method ──────────────────────────────────
# Comment out this method above and uncomment the line below.
# All pkl files must be in the same folder as this notebook.
LOCAL_PATH = os.path.dirname(os.path.abspath("__file__"))
os.chdir(LOCAL_PATH)


In [25]:
# Load preprocessed data (created by Notebook 01)
with open('preprocessed_data.pkl', 'rb') as f:
    data = pickle.load(f)

X_train      = data['X_train']
X_test       = data['X_test']
y_train      = data['y_train']
y_test       = data['y_test']
feature_names = data['feature_names']

print(f"✓ Preprocessed data loaded.")
print(f"  Train: {X_train.shape[0]:,} samples | Test: {X_test.shape[0]:,} samples")
print(f"  Features ({X_train.shape[1]}): {feature_names}")
print("\nNote: Data is already StandardScaler-normalised — ready for SVM.")

✓ Preprocessed data loaded.
  Train: 54,740 samples | Test: 13,685 samples
  Features (13): ['gender', 'height', 'weight', 'ap_hi', 'ap_lo', 'cholesterol', 'gluc', 'smoke', 'alco', 'active', 'age_years', 'bmi', 'bmi_category']

Note: Data is already StandardScaler-normalised — ready for SVM.


## 1. Baseline SVM Model

First, train an SVM with default RBF kernel parameters to get a baseline.
Because the full dataset (~56,000 training samples) is large for SVM, we
use a **stratified subsample of 15,000** for the initial baseline check.

In [26]:
# Stratified subsample for baseline speed test
from sklearn.utils import resample

indices = np.arange(len(X_train))
sample_size = min(15000, len(X_train))
np.random.seed(42)

# Stratified sample
X_sample, y_sample = resample(
    X_train, y_train,
    n_samples=sample_size,
    random_state=42,
    stratify=y_train
)
print(f"Using {sample_size:,} stratified samples for baseline SVM.")
print(f"Class balance in sample: {np.bincount(y_sample) / sample_size}")

Using 15,000 stratified samples for baseline SVM.
Class balance in sample: [0.50546667 0.49453333]


In [27]:
# Baseline SVM – RBF kernel, default C=1, gamma='scale'
svm_baseline = SVC(kernel='rbf', probability=True, random_state=42)
svm_baseline.fit(X_sample, y_sample)

y_pred_base_svm = svm_baseline.predict(X_test)
y_prob_base_svm = svm_baseline.predict_proba(X_test)[:, 1]

# Baseline metrics
print("=== Baseline SVM Performance ===")
print(f"  Accuracy  : {accuracy_score(y_test, y_pred_base_svm):.4f}")
print(f"  Precision : {precision_score(y_test, y_pred_base_svm):.4f}")
print(f"  Recall    : {recall_score(y_test, y_pred_base_svm):.4f}")
print(f"  F1-Score  : {f1_score(y_test, y_pred_base_svm):.4f}")
print(f"  ROC-AUC   : {roc_auc_score(y_test, y_prob_base_svm):.4f}")
print(f"  Support vectors: {svm_baseline.n_support_.sum():,}")

=== Baseline SVM Performance ===
  Accuracy  : 0.7318
  Precision : 0.7565
  Recall    : 0.6746
  F1-Score  : 0.7132
  ROC-AUC   : 0.7851
  Support vectors: 9,098


## 2. Hyperparameter Tuning with GridSearchCV

We tune `C` (regularisation) and `gamma` (kernel width) using **5-Fold Stratified CV**,
optimising for **F1-score**.

| Parameter | Values          | Effect                                          |
|-----------|----------------|-------------------------------------------------|
| `C`       | 0.1, 1, 10, 100 | Low=underfitting, High=overfitting              |
| `gamma`   | 'scale', 0.1, 0.01 | Low=smooth boundary, High=complex boundary |

> **Note:** GridSearchCV is run on a **20,000-sample stratified subset** to manage
> computational cost. The best model is then trained on the **full training set**.

In [28]:
# Stratified subset for GridSearchCV (SVM is slow on large data)
X_gs, y_gs = resample(X_train, y_train, n_samples=8000, random_state=42, stratify=y_train)

# Compact param grid — 3-fold CV × 9 candidates = 27 fits
param_grid_svm = {
    'C':     [0.1, 1, 10],
    'gamma': ['scale', 0.1, 0.01]
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

print("Starting SVM GridSearchCV (3-fold, 9 candidates = 27 fits)...")
svm_grid = GridSearchCV(
    SVC(kernel='rbf', probability=True, random_state=42),
    param_grid_svm,
    cv=cv,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)
svm_grid.fit(X_gs, y_gs)

print(f"\n✓ GridSearchCV complete.")
print(f"  Best C     : {svm_grid.best_params_['C']}")
print(f"  Best gamma : {svm_grid.best_params_['gamma']}")
print(f"  Best CV F1 : {svm_grid.best_score_:.4f}")


Starting SVM GridSearchCV (3-fold, 9 candidates = 27 fits)...
Fitting 3 folds for each of 9 candidates, totalling 27 fits

✓ GridSearchCV complete.
  Best C     : 1
  Best gamma : 0.1
  Best CV F1 : 0.7101


In [29]:
# Train best SVM on FULL training set with best parameters
print("Training best SVM on full training dataset...")
best_svm = SVC(
    kernel='rbf',
    C=svm_grid.best_params_['C'],
    gamma=svm_grid.best_params_['gamma'],
    probability=True,
    random_state=42
)
best_svm.fit(X_train, y_train)
print(f"✓ Best SVM trained on full data.")
print(f"  Number of support vectors: {best_svm.n_support_.sum():,}")
print(f"  Parameters: C={best_svm.C}, gamma={best_svm.gamma}")

Training best SVM on full training dataset...
✓ Best SVM trained on full data.
  Number of support vectors: 32,394
  Parameters: C=1, gamma=0.1


## 3. Final Model Evaluation

In [ ]:
# Predictions from best SVM
y_pred_svm = best_svm.predict(X_test)
y_prob_svm = best_svm.predict_proba(X_test)[:, 1]

svm_metrics = {
    'accuracy':  accuracy_score(y_test, y_pred_svm),
    'precision': precision_score(y_test, y_pred_svm),
    'recall':    recall_score(y_test, y_pred_svm),
    'f1':        f1_score(y_test, y_pred_svm),
    'roc_auc':   roc_auc_score(y_test, y_prob_svm)
}

print("=== SVM (RBF) – Final Test Results ===")
for m, v in svm_metrics.items():
    bar = '█' * int(v * 30)
    print(f"  {m:<12}: {v:.4f}  {bar}")
print("\n=== Detailed Classification Report ===")
print(classification_report(y_test, y_pred_svm, target_names=['No Disease', 'Has Disease']))

### 3.1 Confusion Matrix

In [ ]:
# Confusion Matrix – SVM
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_svm = confusion_matrix(y_test, y_pred_svm)
ConfusionMatrixDisplay(cm_svm, display_labels=['No Disease','Has Disease']
                       ).plot(ax=axes[0], colorbar=False, cmap='Greens')
axes[0].set_title('SVM – Confusion Matrix', fontsize=12, fontweight='bold')

cm_svm_norm = confusion_matrix(y_test, y_pred_svm, normalize='true')
ConfusionMatrixDisplay(cm_svm_norm, display_labels=['No Disease','Has Disease']
                       ).plot(ax=axes[1], colorbar=False, cmap='Greens', values_format='.2f')
axes[1].set_title('SVM – Normalised Confusion Matrix', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('plots/11_svm_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.2 ROC Curve

In [ ]:
# ROC Curve – SVM
fpr_svm, tpr_svm, _ = roc_curve(y_test, y_prob_svm)
auc_svm = roc_auc_score(y_test, y_prob_svm)

plt.figure(figsize=(8, 6))
plt.plot(fpr_svm, tpr_svm, color='#2E7D32', linewidth=2.5,
         label=f'SVM RBF (AUC = {auc_svm:.4f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
plt.fill_between(fpr_svm, tpr_svm, alpha=0.15, color='#2E7D32')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('SVM – ROC Curve', fontsize=13, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('plots/12_svm_roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"ROC-AUC Score: {auc_svm:.4f}")

In [ ]:
# Save SVM model and results
svm_results = {
    'model':        best_svm,
    'best_params':  svm_grid.best_params_,
    'y_pred':       y_pred_svm,
    'y_prob':       y_prob_svm,
    'fpr':          fpr_svm,
    'tpr':          tpr_svm,
    'metrics':      svm_metrics,
}

with open('svm_results.pkl', 'wb') as f:
    pickle.dump(svm_results, f, protocol=4)

print("✓ SVM results saved to 'svm_results.pkl'")
print("\n=== SVM Final Summary ===")
print(f"  Best Parameters : {svm_grid.best_params_}")
for m, v in svm_metrics.items():
    print(f"  {m:<12}: {v:.4f}")

---
## Next Steps
Results saved to `svm_results.pkl`.
Proceed to **04_Results_Comparison.ipynb** for final comparison.
